# Develop and test a QEC protocol (qodec)

Use `qdk` and `qodec` to develop and test a quantum error correction protocol. Start with the C4 code and build a baseline qodec. Then inspect its gadgets and improve them. Along the way, use `qdk`'s built-in profile and audit features to debug and validate.

## Executive summary

```python
code = qodec.Code(stabilizers=..., x=..., z=...)  # define a code
protocol = qdk.ec.build_qodec(c4, strategy=...)  # build a qodec
qdk.ec.audit(protocol)  # validate the qodec
results = qdk.run_qir(qir_program, qodec=protocol, ...)  # simulate a (logical!) program
```

TODO: Plot of the results

## Install

```bash
pip install "qdk[ec]"
```

## 1. Define your code

We will base our protocol on the $4$-qubit $[[4,2,2]]$ error-detecting code C4.
The C4 code stores two logical qubits in four physical qubits. Its two stabilizers are $XXXX$ and $ZZZZ$.

The `qodec` package is used to define the protocol; `qdk` provides the analysis.

In [ ]:
import qodec

c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

Pauli strings in `qodec` use the sparse notation `P_i`, where `P` is one of the Pauli matrices `{X, Y, Z}` and `i` is the index of the qubit on which the Pauli acts.

### Check the code distance

We've defined the code. Now we'd like to make sure it's _correct_. In particular, it should have distance two.  QDK provides a `CodeProfile` that, among other things, can compute the code distance.

In [ ]:
import qdk.ec as ec

code = ec.CodeProfile(c4)
code_distance = code.distance()
print("Code distance:", code_distance)

error = code_distance.witness.product
syndrome = code.syndrome_of(error)
effect = code.logical_effect_of(error)

print(f"Witness: {error}, (syndrome: {list(syndrome)}, logical effect: {effect})")
assert code_distance == error.weight == 2
assert code.is_logical(error)

## 2. Build a baseline qodec

We have a code. But a code alone is a mathematical object: it doesn't describe how to _use_ it on a quantum computer. For that, we need a qodec, a formal description of an executable protocol.

Authoring a full qodec, including its instruction sets and gadgets, can be a substantial task. To get started, QDK provides `build_qodec`, which builds a baseline protocol from a code definition. There are many ways to do this; the `strategy` parameter selects the recipe.

We will use `bare-css/v1`, a strategy that measures syndromes with ancillas but adds no flag qubits.

The result has two layers: logical C4 operations and physical Stim operations. Each logical instruction has a _gadget_: a circuit that implements it using the next layer. Its encodings identify the physical qubits for each logical block; its equations describe checks and readouts.

Let's build the qodec and inspect the start of its YAML representation. Then we'll use `gadgets` to edit the protocol directly.

In [ ]:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
print(protocol.dumps()[:500], "...")

### Inspect `prepare` instructions and gadgets

Let's look at `prepare_z_all` and its implementation together. The instruction produces one C4 block in logical $|00\rangle$. It has no input block and one output block, which encodes two logical qubits. The `stabilize` action requires both logical Z operators, `Z_0` and `Z_1`, to have eigenvalue +1.

In [ ]:
instruction_set = protocol.layers[0].instruction_set
instruction_set.instructions["prepare_z_all"]

The instruction specifies the contract but does _not_ supply the implementation. That is the gadget's responsibility.  The corresponding `prepare_z_all` gadget is shown below.
It includes a circuit and parity checks. A _check_ is an equation that should evaluate to zero without faults. The equations include measurement bits and input/output encoding signs.

In [ ]:
gadgets = protocol.layers[0].gadgets
gadgets["prepare_z_all"]

The circuit source is Stim (qodec supports any circuit representation with an associated parser). So we can use `stim` directly to look at the circuit.

In [ ]:
import stim
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

Does this circuit prepare the state that the instruction claims it does? As with `CodeProfile`, we can use a `GadgetProfile` to analyze it. Its `objective` is the declared instruction; its `action` is the behavior computed from the circuit. Let's inspect the computed action and check that it matches the instruction.

In [ ]:
from IPython.display import Markdown, display


def display_gadget_action(profile: ec.GadgetProfile) -> None:
    objective, action = profile.objective, profile.action
    if objective is None:
        result = "**Comparison unavailable:** no declared operation."
    elif action.is_equivalent_to(objective):
        result = "**Meets objective:** ✓"
    else:
        result = f"**Meets objective:** ✗\n\n{objective.why_not_equivalent_to(action)}"
    display(Markdown(f"**Computed action**\n\n```text\n{action}\n```\n\n{result}"))


preparation_profile = ec.GadgetProfile(gadgets["prepare_z_all"])
display_gadget_action(preparation_profile)

### Inspect `cx`

Now let's look at `cx_all`. It takes two C4 blocks and applies CNOT between corresponding logical qubits. The `action` treats qubit indices as flat. The first C4 block corresponds to indices 0 and 1. The second C4 block corresponds to indices 2 and 3.

In [ ]:
instruction_set.instructions["cx_all"]

Now, let's inspect the corresponding CNOT gadget. 

In [ ]:
gadgets["cx_all"]

The circuit is very simple: just transversal CNOTs across the two 4-qubit physical blocks. The `in/out` fields specify how to connect circuit labels to the logical blocks.
Note that, even though this gadget has no readouts, it _does_ include parity checks. The parity checks here relate the input stabilizers to the output stabilizers.

Next, we can look at the action of the gadget and verify that it implements the intended instruction.

In [ ]:
cnot_profile = ec.GadgetProfile(gadgets["cx_all"])
display_gadget_action(cnot_profile)

## 3. Check the entire qodec for correctness

We've inspected two instructions and their gadgets. Now we'd like to check the whole protocol. QDK provides `audit` to compare the circuits with their declared operations and equations. The audit should report no errors or warnings for the generated protocol. 

To see what the audit can tell us, we will deliberately remove some equations. The circuits will stay the same. Then we'll use QDK to derive the missing equations and check the protocol again.

In [ ]:
report = ec.audit(protocol)
print(report)

### Remove the measurement readouts

The `measure_x_all` gadget should report two logical measurement results. Its readout equations tell us which physical measurement bits and frame signs to combine for each result.

Let's clear those equations. The circuit still measures the qubits, but the protocol no longer defines its logical results. The audit should report two `gadget/missing-observable` errors and show where the equations are missing.

In [ ]:
gadgets["measure_x_all"].readouts.clear()
print(ec.audit(protocol))

### Remove the CNOT checks

Now let's remove the four checks from `cx_all`. These relate the input and output stabilizer signs. The circuit still performs the logical CNOT, but we have removed the equations that specify its output stabilizer signs. The audit should add four `gadget/incomplete-output-frame` warnings to the two readout errors.

In [ ]:
gadgets["cx_all"].checks.clear()
print(ec.audit(protocol))

Replacing the missing gadget information can be tedious.  For situations like these, QDK offers `filled`.  This function accepts a partial qodec (or gadget) and attempts to fill it in based on the existing contents.

In [ ]:
protocol = ec.filled(protocol)
gadgets = protocol.layers[0].gadgets
print(ec.audit(protocol))

## 4. Check the gadgets for fault tolerance

The qodec passes the audit. But the audit doesn't consider fault tolerance. What happens when a circuit has faults?

We checked the code distance in section 1. Now we will use `GadgetProfile.distance()` to check the _gadget distance_: the fewest allowed circuit faults that change the logical action, stay within the output codespaces, and leave every check and flag zero.

The default model is circuit noise: a Pauli error after a call, a flip of that call's recorded readout bits, or both.

Let's compute the distance of each gadget. Each result includes a _witness_: a set of faults that achieves the reported distance.

In [ ]:
def report_gadget_distances(protocol: qodec.Qodec) -> None:
    distances = {
        mnemonic: ec.GadgetProfile(gadget).distance()
        for mnemonic, gadget in protocol.layers[0].gadgets.items()
    }
    rows = [
        "| Gadget | Distance | Witness |",
        "| --- | ---: | --- |",
    ]
    rows.extend(
        f"| `{mnemonic}` | {distance} | {distance.witness} |"
        for mnemonic, distance in sorted(distances.items())
    )
    display(Markdown("\n".join(rows)))

report_gadget_distances(protocol)

We're using a distance-two code, so we might hope that our gadgets also achieve distance two. Some of them do, but the `prepare` and `syndrome` gadgets do not. We fix that in the next section.

## 5. Increase the gadget distances

### Why the syndrome gadget has distance one

Let's inspect the `syndrome` circuit and the distance-one witness.


In [ ]:
from IPython.display import HTML, display

circuit = gadgets["syndrome"].circuit
diagram = str(stim.Circuit(circuit.source).diagram("timeline-svg-html"))
call_text = "\n".join(
    f"{index}: {call.mnemonic} {call.operands}"
    for index, call in enumerate(circuit.calls())
)

display(HTML(
    '<div style="display:flex;gap:2rem;flex-wrap:wrap">'
    f'<pre>{diagram}</pre>'
    f'<pre>{call_text}</pre>'
    '</div>'
))


Ancilla 4 measures the X stabilizer with CNOTs. Ancilla 5 then measures the Z stabilizer with controlled-Z gates. We will follow faults on these ancillas through the remaining gates and inspect which checks and output signs they change.

In [ ]:
idle_profile = ec.GadgetProfile(gadgets["syndrome"])
calls = gadgets["syndrome"].circuit.calls()
distance = idle_profile.distance()
location = distance.witness.product.locations[0]
call = calls[location.after_call]
effect = idle_profile.effects_of([distance.witness.product])[0]
print(f"{distance.witness} ({call.mnemonic} {call.operands}) → {effect}")

The offending fault occurs on the third call of the gadget, `CX 4 1`. It spreads through subsequent `CX` gates, ultimately flipping the output `z[0]` observable, but _not_ flipping the Z-stabilizer measurement. It is a logical `X_0` error.

But that's not the only problematic fault. The distance has several witnesses, which we examine below.

In [ ]:
faults = [witness.product for witness in distance.witnesses]
effects = idle_profile.effects_of(faults)

for fault, effect in zip(faults, effects):
    location = fault.locations[0]
    call = calls[location.after_call]
    print(f"{fault}\t({call.mnemonic} {call.operands})\t→ {effect}")

### Improve the syndrome circuit

All of the problematic faults are errors that occur on an ancilla qubit and then spread to multiple data qubits and go undetected. We need a circuit that avoids this phenomenon.
In this case, a circuit by [Reichardt](https://arxiv.org/pdf/1804.06995#page=4) does the trick. It interleaves the ancilla interactions so that any dangerous error spreads to the other ancilla's measurement result. It uses eight CNOTs and the same two ancillas, without adding a flag qubit.

Let's replace our circuit with Reichardt's. Then we'll use `filled` to recompute the parity checks and evaluate the new gadget distance.

In [ ]:
reichardt_source = """
R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
display(stim.Circuit(reichardt_source).diagram("timeline-svg-html"))

gadgets["syndrome"].circuit.source = reichardt_source
gadgets["syndrome"] = ec.filled(gadgets["syndrome"])

improved_syndrome_profile = ec.GadgetProfile(gadgets["syndrome"])
print("Improved syndrome distance:", improved_syndrome_profile.distance())
print(ec.audit(protocol))

### Detect analogous hook faults

The reordered circuit still lets an ancilla fault spread to two data qubits, but the other ancilla now detects it. The gadget distance of two is proof.

But it's possible to see this improvement explicitly by looking at how previously dangerous ancilla faults impact the new circuit.

In [ ]:
improved_calls = gadgets["syndrome"].circuit.calls()
faults = [
    ec.FaultEvent.after(5, ec.Pauli("Z_5")),  
    ec.FaultEvent.after(7, ec.Pauli("X_4"))
]
effects = improved_syndrome_profile.effects_of(faults)

for fault, effect in zip(faults, effects):
    call = improved_calls[fault.locations[0].after_call]
    print(f"{fault} ({call.mnemonic} {call.operands}) →")
    for reference in effect.checks + effect.readouts:
        equation = gadgets["syndrome"].resolve(reference).value(tuple)
        print(f"  {reference} = {[str(term) for term in equation]}")
    for reference in effect.frames:
        print(f"  {reference}")
    print()

The faults still flip logical operators (`Z_5` flips `out[0].x[0]`, `X_4` flips `out[0].z[1]`). But now, those faults are detected by parity checks, thereby increasing the distance.

### Improve both preparations with a flag

The `syndrome` gadget now looks good. We also need to improve the `prepare` gadgets. Let's look at the $|00\rangle$ circuit.

In [ ]:
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

This circuit is nearly identical to our original `syndrome` circuit, and so it's subject to the same problems: a fault on an ancilla spreads to an undetected logical error.

We'll fix this by instead using a unitary encoding circuit and a _flag_. A flag is a special parity check that gets promoted to a readout. It can then be used to reject noisy states, a process known as pre-selection.
Below, we add a `reject` flag to each gadget and bind it to the measurement. The `GadgetProfile.distance` calculation accounts for the flag automatically.

The $|++\rangle$ preparation is identical to the $|00\rangle$ preparation followed by transversal Hadamard gates.


In [ ]:
flagged_preparation_source = """R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
"""
preparation_suffixes = {
    "prepare_z_all": "M 4\n",
    "prepare_x_all": "H 0 1 2 3\nM 4\n",
}
for mnemonic, suffix in preparation_suffixes.items():
    gadget = gadgets[mnemonic]
    gadget.implements.flags = ["reject"]
    gadget.circuit.source = flagged_preparation_source + suffix
    gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]
    gadgets[mnemonic] = ec.filled(gadget)
    print(mnemonic, "distance:", ec.GadgetProfile(gadgets[mnemonic]).distance())
print(ec.audit(protocol))

In [ ]:
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

## 6. Inspect and save the improved qodec

We've changed the syndrome and preparation circuits. Now let's check the whole qodec again. The audit should report no errors or warnings, and all six gadgets should have distance two.

In [ ]:
protocol.description = (
    "C4 built with bare-css/v1, with self-checking syndrome extraction "
    "and single-flag preparation circuits."
)
print(ec.audit(protocol))
report_gadget_distances(protocol);

### Save the qodec

We now have a qodec we'd like to keep. Use `save` to write its circuits and declarations into one YAML bundle.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    saved_path = protocol.save(directory, single_file=True)
    assert qodec.Qodec.load(saved_path) == protocol
print("Round trip: OK")

## 7. Evaluate performance

Before measuring performance, try one noiseless shot through `qdk.simulation.run_qir`. The Q# program below prepares one logical qubit in $|0\rangle$ and measures it. These cells expose the remaining QIR binding limitation; they are not yet a performance benchmark.

In [ ]:
from qdk import TargetProfile, qsharp
from qdk.simulation import run_qir

qsharp.init(target_profile=TargetProfile.Base)
qir_program = qsharp.compile("{ use q = Qubit(); M(q) }")

`build_qodec` supplies the C4 binding, and `filled` preserves it. Run the protocol from section 6 without a registry workaround. The call still raises `UnboundOperation: ISA 'C4' does not implement 'prepare'`: QIR requests a single-qubit preparation, while `prepare_z_all` prepares both logical qubits in a C4 block. The binder also excludes instructions with flags, including our improved preparations. We leave the exception visible and use `on_shot_failure="raise"` so shot failures are not discarded.

In [ ]:
results = run_qir(
    qir_program,
    shots=1,
    seed=7,
    qodec=protocol,
    on_shot_failure="raise",
)
results

## What we established

We started with the C4 code and built a baseline qodec. We used the audit to find missing equations, then used gadget profiles to find faults and test better circuits. The resulting six gadgets pass the audit and each has distance two. We can also save and load the protocol without changing its declarations.

These results apply to individual gadgets under our Pauli and readout-fault model. To assess a complete implementation, we would still need to account for hardware connectivity, timing, additional fault locations, correlated noise, and how the gadgets work together.